# SBERT + CatBoost Fusion for Duplicate Detection

Notebook for training SBERT on text pairs and a CatBoost classifier on top of SBERT cosine similarity plus engineered features.

In [ ]:
!pip install catboost

In [ ]:
import json
import math
import random
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from sentence_transformers import InputExample, SentenceTransformer
from sentence_transformers.losses import CosineSimilarityLoss
from sentence_transformers.evaluation import BinaryClassificationEvaluator

/tmp/ipykernel_634/1402053451.py:22: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import CosineSimilarityLoss
/tmp/ipykernel_634/1402053451.py:23: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import BinaryClassificationEvaluator


In [ ]:
EXTRA_FEATURES = [
    "char_sim",
    "norm_sim",
    "jaccard",
    "len_diff",
    "tok_diff",
    "same_action",
    "same_domain",
]

args = SimpleNamespace(
    csv_path=Path("pairs_full_processed.csv"),
    hard_set_path=None,
    text_col_1="event_1",
    text_col_2="event_2",
    label_col="label",
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    output_dir=Path("artifacts/sbert_catboost_fusion"),
    epochs=2,
    batch_size=64,
    eval_batch_size=256,
    learning_rate=2e-5,
    max_samples=0,
    test_size=0.1,
    val_size=0.1,
    seed=42,
    keep_underscores=False,
    catboost_iterations=500,
    catboost_depth=6,
    catboost_learning_rate=0.05,
    catboost_loss_function="Logloss",
    catboost_eval_metric="F1",
)

args

namespace(csv_path=PosixPath('pairs_full_processed.csv'),
          hard_set_path=None,
          text_col_1='event_1',
          text_col_2='event_2',
          label_col='label',
          model_name='sentence-transformers/all-MiniLM-L6-v2',
          output_dir=PosixPath('artifacts/sbert_catboost_fusion'),
          epochs=2,
          batch_size=64,
          eval_batch_size=256,
          learning_rate=2e-05,
          max_samples=0,
          test_size=0.1,
          val_size=0.1,
          seed=42,
          keep_underscores=False,
          catboost_iterations=500,
          catboost_depth=6,
          catboost_learning_rate=0.05,
          catboost_loss_function='Logloss',
          catboost_eval_metric='F1')

In [ ]:
def validate_args(args) -> None:
    if not args.csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {args.csv_path}")

    if args.hard_set_path is not None and not Path(args.hard_set_path).exists():
        raise FileNotFoundError(f"Hard set file not found: {args.hard_set_path}")

    if args.max_samples < 0:
        raise ValueError("max_samples must be >= 0")

    if not (0.0 < args.test_size < 1.0):
        raise ValueError("test_size must be between 0 and 1")

    if not (0.0 < args.val_size < 1.0):
        raise ValueError("val_size must be between 0 and 1")

    if args.test_size + args.val_size >= 1.0:
        raise ValueError("test_size + val_size must be < 1")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_text(value: str, keep_underscores: bool) -> str:
    text = str(value).strip().lower()
    if not keep_underscores:
        text = text.replace("_", " ")
    return " ".join(text.split())


def print_metrics(title: str, metrics: dict) -> None:
    print(f"\n{'=' * 60}")
    print(title)
    print(f"{'=' * 60}")
    print(f"Threshold : {metrics['threshold']:.4f}")
    print(f"Accuracy  : {metrics['accuracy']:.4f}")
    print(f"Precision : {metrics['precision']:.4f}")
    print(f"Recall    : {metrics['recall']:.4f}")
    print(f"F1        : {metrics['f1']:.4f}")
    print()
    print(metrics["classification_report"])


def strip_report(metrics: dict | None) -> dict | None:
    if metrics is None:
        return None
    return {k: v for k, v in metrics.items() if k != "classification_report"}

In [ ]:
def load_dataset(path: Path, args, require_extra_features: bool = False) -> pd.DataFrame:
    df = pd.read_csv(path)

    required = {args.text_col_1, args.text_col_2, args.label_col}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    missing_extra = [f for f in EXTRA_FEATURES if f not in df.columns]
    if require_extra_features and missing_extra:
        raise ValueError(f"Missing extra feature columns: {missing_extra}")

    extra_present = [f for f in EXTRA_FEATURES if f in df.columns]
    cols_to_keep = [args.text_col_1, args.text_col_2, args.label_col] + extra_present
    df = df[cols_to_keep].copy().dropna().reset_index(drop=True)

    df[args.label_col] = df[args.label_col].astype(int)
    df[args.text_col_1] = df[args.text_col_1].map(
        lambda x: normalize_text(x, args.keep_underscores)
    )
    df[args.text_col_2] = df[args.text_col_2].map(
        lambda x: normalize_text(x, args.keep_underscores)
    )

    for feature in extra_present:
        df[feature] = pd.to_numeric(df[feature], errors="raise")

    if args.max_samples > 0:
        df = (
            df.groupby(args.label_col, group_keys=False)
            .apply(
                lambda chunk: chunk.sample(
                    n=min(args.max_samples, len(chunk)),
                    random_state=args.seed,
                )
            )
            .sample(frac=1.0, random_state=args.seed)
            .reset_index(drop=True)
        )

    print(
        f"Dataset: {len(df)} rows | positive rate: {df[args.label_col].mean():.3f} | "
        f"extra features: {extra_present}"
    )
    return df


def load_hard_set(path: Path, args, expected_feature_cols: list[str]) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.rename(columns={"event_1": args.text_col_1, "event_2": args.text_col_2})

    required = {args.text_col_1, args.text_col_2, args.label_col, *expected_feature_cols}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Hard set is missing columns: {sorted(missing)}")

    cols_to_keep = [args.text_col_1, args.text_col_2, args.label_col] + expected_feature_cols
    df = df[cols_to_keep].copy().dropna().reset_index(drop=True)
    df[args.label_col] = df[args.label_col].astype(int)
    df[args.text_col_1] = df[args.text_col_1].map(
        lambda x: normalize_text(x, args.keep_underscores)
    )
    df[args.text_col_2] = df[args.text_col_2].map(
        lambda x: normalize_text(x, args.keep_underscores)
    )

    for feature in expected_feature_cols:
        df[feature] = pd.to_numeric(df[feature], errors="raise")

    print(
        f"Hard set: {len(df)} rows | positive rate: {df[args.label_col].mean():.3f} | "
        f"features: {expected_feature_cols}"
    )
    return df


def build_examples(df: pd.DataFrame, text_col_1: str, text_col_2: str, label_col: str) -> list[InputExample]:
    return [
        InputExample(texts=[row[text_col_1], row[text_col_2]], label=float(row[label_col]))
        for _, row in df.iterrows()
    ]

In [ ]:
class CatBoostFusionClassifier:
    """
    CatBoost on top of [SBERT cosine similarity] + engineered features.
    Trained on train split, threshold calibrated on val split.
    """

    def __init__(self, args) -> None:
        self.feature_cols: list[str] = []
        self.model = CatBoostClassifier(
            iterations=args.catboost_iterations,
            depth=args.catboost_depth,
            learning_rate=args.catboost_learning_rate,
            loss_function=args.catboost_loss_function,
            eval_metric=args.catboost_eval_metric,
            random_seed=args.seed,
            verbose=False,
        )

    def _build_matrix(self, cosine_scores: np.ndarray, df: pd.DataFrame) -> pd.DataFrame:
        data = {"sbert_cosine": cosine_scores}
        for feature in self.feature_cols:
            if feature not in df.columns:
                raise ValueError(f"Missing fusion feature column: {feature}")
            data[feature] = df[feature].to_numpy(dtype=float)
        return pd.DataFrame(data, index=df.index)

    def fit(
        self,
        train_cosine: np.ndarray,
        train_df: pd.DataFrame,
        train_labels: np.ndarray,
        val_cosine: np.ndarray,
        val_df: pd.DataFrame,
        val_labels: np.ndarray,
    ) -> None:
        self.feature_cols = [f for f in EXTRA_FEATURES if f in train_df.columns]
        if not self.feature_cols:
            raise ValueError("No extra feature columns found for fusion training.")

        train_X = self._build_matrix(train_cosine, train_df)
        val_X = self._build_matrix(val_cosine, val_df)

        self.model.fit(
            train_X,
            train_labels,
            eval_set=(val_X, val_labels),
            use_best_model=True,
            early_stopping_rounds=50
            )
        print(f"CatBoostFusionClassifier trained with features: ['sbert_cosine'] + {self.feature_cols}")

    def predict_proba(self, cosine_scores: np.ndarray, df: pd.DataFrame) -> np.ndarray:
        X = self._build_matrix(cosine_scores, df)
        return self.model.predict_proba(X)[:, 1]

    def save(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        self.model.save_model(str(path))

In [ ]:
def encode_pairs(model: SentenceTransformer, df: pd.DataFrame, args) -> tuple[np.ndarray, np.ndarray]:
    emb1 = model.encode(
        df[args.text_col_1].tolist(),
        batch_size=args.eval_batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    emb2 = model.encode(
        df[args.text_col_2].tolist(),
        batch_size=args.eval_batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    cosine_scores = torch.nn.functional.cosine_similarity(emb1, emb2).cpu().numpy()
    labels = df[args.label_col].to_numpy()
    return cosine_scores, labels


def find_best_threshold(scores: np.ndarray, labels: np.ndarray, low: float, high: float) -> tuple[float, float]:
    best_threshold, best_f1 = low, -1.0
    for threshold in np.linspace(low, high, 401):
        preds = (scores >= threshold).astype(int)
        score = f1_score(labels, preds, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_threshold = float(threshold)
    return best_threshold, best_f1


def evaluate_with_threshold(scores: np.ndarray, labels: np.ndarray, threshold: float) -> dict:
    preds = (scores >= threshold).astype(int)
    return {
        "threshold": threshold,
        "accuracy": float(accuracy_score(labels, preds)),
        "precision": float(precision_score(labels, preds, zero_division=0)),
        "recall": float(recall_score(labels, preds, zero_division=0)),
        "f1": float(f1_score(labels, preds, zero_division=0)),
        "classification_report": classification_report(labels, preds, digits=4, zero_division=0),
    }

In [ ]:
validate_args(args)
set_seed(args.seed)
args.output_dir.mkdir(parents=True, exist_ok=True)

df = load_dataset(args.csv_path, args, require_extra_features=True)

train_df, test_df = train_test_split(
    df,
    test_size=args.test_size,
    random_state=args.seed,
    stratify=df[args.label_col],
)
relative_val_size = args.val_size / (1.0 - args.test_size)
train_df, val_df = train_test_split(
    train_df,
    test_size=relative_val_size,
    random_state=args.seed,
    stratify=train_df[args.label_col],
)

print(f"Split -> train: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}")

Dataset: 50000 rows | positive rate: 0.500 | extra features: ['char_sim', 'norm_sim', 'jaccard', 'len_diff', 'tok_diff', 'same_action', 'same_domain']
Split -> train: 40000  val: 5000  test: 5000


In [ ]:
model = SentenceTransformer(args.model_name)
train_examples = build_examples(train_df, args.text_col_1, args.text_col_2, args.label_col)
train_loader = DataLoader(train_examples, shuffle=True, batch_size=args.batch_size)
train_loss = CosineSimilarityLoss(model=model)

evaluator = BinaryClassificationEvaluator(
    sentences1=val_df[args.text_col_1].tolist(),
    sentences2=val_df[args.text_col_2].tolist(),
    labels=val_df[args.label_col].astype(bool).tolist(),
    batch_size=args.eval_batch_size,
    show_progress_bar=True,
    write_csv=True,
    name="validation",
)

warmup_steps = math.ceil(len(train_loader) * args.epochs * 0.1)
model.fit(
    train_objectives=[(train_loader, train_loss)],
    evaluator=evaluator,
    epochs=args.epochs,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": args.learning_rate},
    output_path=str(args.output_dir),
    save_best_model=True,
    show_progress_bar=True,
)

best_model = SentenceTransformer(str(args.output_dir))
print("Best SBERT checkpoint loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Best SBERT checkpoint loaded.


In [ ]:
from sentence_transformers import SentenceTransformer

best_model = SentenceTransformer("artifacts/sbert_catboost_fusion")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
print("Encoding train...")
train_cosine, train_labels = encode_pairs(best_model, train_df, args)

print("Encoding val...")
val_cosine, val_labels = encode_pairs(best_model, val_df, args)

print("Encoding test...")
test_cosine, test_labels = encode_pairs(best_model, test_df, args)

Encoding train...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Encoding val...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Encoding test...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
sbert_threshold, sbert_val_f1 = find_best_threshold(val_cosine, val_labels, low=-1.0, high=1.0)
test_metrics_sbert = evaluate_with_threshold(test_cosine, test_labels, sbert_threshold)

print(f"SBERT-only threshold: {sbert_threshold:.4f} | val F1: {sbert_val_f1:.4f}")
print_metrics("SBERT only - TEST", test_metrics_sbert)

SBERT-only threshold: 0.4800 | val F1: 0.9970

SBERT only - TEST
Threshold : 0.4800
Accuracy  : 0.9946
Precision : 0.9920
Recall    : 0.9972
F1        : 0.9946

              precision    recall  f1-score   support

           0     0.9972    0.9920    0.9946      2500
           1     0.9920    0.9972    0.9946      2500

    accuracy                         0.9946      5000
   macro avg     0.9946    0.9946    0.9946      5000
weighted avg     0.9946    0.9946    0.9946      5000



In [ ]:
fusion_clf = CatBoostFusionClassifier(args)
fusion_clf.fit(
    train_cosine, train_df, train_labels,
    val_cosine, val_df, val_labels,
)

val_fusion_proba = fusion_clf.predict_proba(val_cosine, val_df)
fusion_threshold, fusion_val_f1 = find_best_threshold(val_fusion_proba, val_labels, low=0.0, high=1.0)

test_fusion_proba = fusion_clf.predict_proba(test_cosine, test_df)
test_metrics_fusion = evaluate_with_threshold(test_fusion_proba, test_labels, fusion_threshold)

fusion_model_path = args.output_dir / "catboost_fusion.cbm"
fusion_clf.save(fusion_model_path)

print(f"Fusion threshold: {fusion_threshold:.4f} | val F1: {fusion_val_f1:.4f}")
print_metrics("SBERT + CatBoost features - TEST", test_metrics_fusion)
print(f"CatBoost model saved to: {fusion_model_path.resolve()}")

CatBoostFusionClassifier trained with features: ['sbert_cosine'] + ['char_sim', 'norm_sim', 'jaccard', 'len_diff', 'tok_diff', 'same_action', 'same_domain']
Fusion threshold: 0.3325 | val F1: 0.9992

SBERT + CatBoost features - TEST
Threshold : 0.3325
Accuracy  : 0.9986
Precision : 0.9980
Recall    : 0.9992
F1        : 0.9986

              precision    recall  f1-score   support

           0     0.9992    0.9980    0.9986      2500
           1     0.9980    0.9992    0.9986      2500

    accuracy                         0.9986      5000
   macro avg     0.9986    0.9986    0.9986      5000
weighted avg     0.9986    0.9986    0.9986      5000

CatBoost model saved to: /content/drive/MyDrive/sbert_model/catboost_fusion.cbm


In [ ]:
hard_metrics_sbert = None
hard_metrics_fusion = None

if args.hard_set_path is not None:
    hard_df = load_hard_set(Path(args.hard_set_path), args, fusion_clf.feature_cols)
    print("Encoding hard set...")
    hard_cosine, hard_labels = encode_pairs(best_model, hard_df, args)

    hard_metrics_sbert = evaluate_with_threshold(hard_cosine, hard_labels, sbert_threshold)
    hard_fusion_proba = fusion_clf.predict_proba(hard_cosine, hard_df)
    hard_metrics_fusion = evaluate_with_threshold(hard_fusion_proba, hard_labels, fusion_threshold)

    print_metrics("SBERT only - HARD SET", hard_metrics_sbert)
    print_metrics("SBERT + CatBoost features - HARD SET", hard_metrics_fusion)
else:
    print("Hard set path not provided - skipping.")

Hard set path not provided - skipping.


In [ ]:
summary = {
    "dataset": {
        "csv_path": str(args.csv_path),
        "rows_total": int(len(df)),
        "rows_train": int(len(train_df)),
        "rows_val": int(len(val_df)),
        "rows_test": int(len(test_df)),
        "positive_rate_total": float(df[args.label_col].mean()),
        "extra_features_used": fusion_clf.feature_cols,
    },
    "training": {
        "base_model": args.model_name,
        "epochs": args.epochs,
        "batch_size": args.batch_size,
        "learning_rate": args.learning_rate,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    },
    "catboost": {
        "iterations": args.catboost_iterations,
        "depth": args.catboost_depth,
        "learning_rate": args.catboost_learning_rate,
        "loss_function": args.catboost_loss_function,
        "eval_metric": args.catboost_eval_metric,
    },
    "sbert_only": {
        "val_threshold": sbert_threshold,
        "val_f1": sbert_val_f1,
        "test": strip_report(test_metrics_sbert),
        "hard": strip_report(hard_metrics_sbert),
    },
    "sbert_catboost_fusion": {
        "val_threshold": fusion_threshold,
        "val_f1": fusion_val_f1,
        "test": strip_report(test_metrics_fusion),
        "hard": strip_report(hard_metrics_fusion),
    },
}

metrics_path = args.output_dir / "metrics.json"
metrics_path.write_text(json.dumps(summary, ensure_ascii=True, indent=2), encoding="utf-8")

comparison_rows = [
    ("SBERT only", test_metrics_sbert, hard_metrics_sbert),
    ("SBERT + CatBoost", test_metrics_fusion, hard_metrics_fusion),
]

print("\nFinal comparison")
print(f"{'Model':<22} {'Test F1':>10} {'Hard F1':>10}")
print("-" * 44)
for name, test_metrics, hard_metrics in comparison_rows:
    hard_f1 = f"{hard_metrics['f1']:.4f}" if hard_metrics else "N/A"
    print(f"{name:<22} {test_metrics['f1']:>10.4f} {hard_f1:>10}")

print(f"\nSBERT model dir: {args.output_dir.resolve()}")
print(f"Metrics saved to: {metrics_path.resolve()}")


Final comparison
Model                     Test F1    Hard F1
--------------------------------------------
SBERT only                 0.9946        N/A
SBERT + CatBoost           0.9986        N/A

SBERT model dir: /content/drive/MyDrive/sbert_model
Metrics saved to: /content/drive/MyDrive/sbert_model/metrics.json


In [ ]:
#Проверка

In [ ]:
import torch

def predict_duplicate_sbert(text1, text2, model, threshold=0.47):
    emb1 = model.encode([text1], convert_to_tensor=True, normalize_embeddings=True)
    emb2 = model.encode([text2], convert_to_tensor=True, normalize_embeddings=True)

    cosine = torch.nn.functional.cosine_similarity(emb1, emb2).item()
    pred = int(cosine >= threshold)

    return {
        "cosine": cosine,
        "is_duplicate": bool(pred)
    }

In [ ]:
predict_duplicate_sbert(
    "payment failed",
    "transaction error",
    best_model,
    threshold=sbert_threshold
)

{'cosine': 0.6698234677314758, 'is_duplicate': True}